# 02. Data Validation & Quality Audit

This notebook executes a comprehensive data quality audit on the raw S&P 500 panel dataset extracted in the previous module. It evaluates temporal integrity, missingness patterns, trading period boundaries, structural index consistency, and anomalous market states (such as zero volume or impossible OHLC values) to establish whether pre-processing or cleaning transformations are required.

## 1. Environment & Setup


### 1.1 Module Imports & Environment Configuration

Load required dependencies, configure dynamic path insertion for the src module, and import validation utility functions.

In [1]:
import sys
from pathlib import Path
import pandas as pd 

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

# Validation functions
from src.data.validate_data import (
    check_yfinance_dtypes,
    count_nans_by_ticker,
    create_data_quality_summary,
    describe_yfinance_data,
    detect_impossible_values,
    get_trading_periods_by_ticker,
    has_duplicate_dates,
    is_index_sorted,
    summarize_date_gaps,
)

## 2. Data Quality Audit & Empirical Diagnostics

### 2.1 General Dataset Profiling

Load the raw price matrix and inspect overarching properties including ticker coverage, temporal boundaries, and total trading sessions.

In [2]:
prices = pd.read_parquet("../data/raw/sp500_prices.parquet")

info = describe_yfinance_data(prices)
print(info)

{'n_assets': 499, 'n_observations': 3773, 'start_date': Timestamp('2010-01-04 00:00:00'), 'end_date': Timestamp('2024-12-30 00:00:00')}


The panel spans **499** tickers across **3,773 trading days** from **2010-01-04** to **2024-12-31**. The ticker count reflects the initial 503 constituent universe minus 4 tickers flagged during extraction.

### 2.2 Temporal Continuity & Trading Gap Distribution

Evaluate daily cadence by analyzing day-over-day date delta distributions to verify trading calendar alignment and detect missing sessions.

In [3]:
date_gaps = summarize_date_gaps(prices)
print(date_gaps)

1 day     2955
2 days      35
3 days     680
4 days     101
5 days       1
Name: count, dtype: int64


Date gaps consist almost entirely of 1-day step-changes (consecutive trading days) and 3-day steps (weekend gaps), with a single edge gap exceeding 4 days.

> **Decision**: The raw dataset confirms a clean, continuous daily resolution. No adjustments to frequency alignment are required at this stage.

### 2.3 Missingness Profiling & Active Trading Window Analysis

Analyze NaN occurrence per constituent to distinguish between systemic data loss and natural Initial Public Offering (IPO) entry dates.

In [4]:
nan_counts = count_nans_by_ticker(prices)
display(nan_counts)

,total_nans,internal_nans,nans_outside_trading_period
Ticker,,,
GEV,21486,0,21486
SOLV,21480,0,21480
RDDT,21462,0,21462
VLTO,20766,0,20766
KVUE,20136,0,20136
...,...,...,...
AES,0,0,0
AFL,0,0,0
AIG,0,0,0


In [5]:
trading_periods = get_trading_periods_by_ticker(prices)

full_period_tickers = trading_periods["full_period"]
partial_period_tickers = trading_periods["partial_period"]

print(f"Full-period coverage count: {len(full_period_tickers)}")
print(f"Partial-period coverage count: {len(partial_period_tickers)}")


Full-period coverage count: 419
Partial-period coverage count: 80


419 tickers cover the full 15-year sample (2010–2024). The remaining 80 tickers exhibit missing values exclusively before their respective listing dates, with 100% reaching the final sample date (2024-12-31). Zero internal NaN values were detected during active trading periods.

> **Decision**: Missing values are purely pre-IPO entries. All tickers will be retained in the raw panel; survivorship-bias and dynamic universe eligibility will be handled downstream at the factor construction stage.

### 2.4 Structural Integrity & Market Anomaly Detection

Validate index monotonicity, verify the absence of duplicated date records, test OHLC price relationships, check for negative volume, and flag zero-volume trading days.

In [6]:
print(has_duplicate_dates(prices))  # True if duplicated dates exist
print(is_index_sorted(prices))      # True if the index is chronologically sorted

False
True


In [7]:
invalid_values = detect_impossible_values(prices)
display(invalid_values)

# Inspect companies with sporadic zero volume days
display(invalid_values[invalid_values['zero_volume'] != 0].tail())

n_zero_vol = (invalid_values["zero_volume"] > 0).sum()
print(f"Companies with at least one zero-volume day: {n_zero_vol}")

,invalid_open,invalid_high,invalid_low,invalid_close,invalid_adj_close,zero_volume,negative_volume,low_greater_than_high,high_less_than_open,high_less_than_close,low_greater_than_open,low_greater_than_close,total_issues
Ticker,,,,,,,,,,,,,
SW,0,0,0,0,0,2409,0,0,0,0,0,0,2409
FERG,0,0,0,0,0,2079,0,0,0,0,0,0,2079
AMCR,0,0,0,0,0,1371,0,0,0,0,0,0,1371
HWM,0,0,0,0,0,35,0,0,0,0,0,0,35
CHTR,0,0,0,0,0,18,0,0,0,0,0,0,18
...,...,...,...,...,...,...,...,...,...,...,...,...,...
AES,0,0,0,0,0,0,0,0,0,0,0,0,0
AFL,0,0,0,0,0,0,0,0,0,0,0,0,0
AIG,0,0,0,0,0,0,0,0,0,0,0,0,0


,invalid_open,invalid_high,invalid_low,invalid_close,invalid_adj_close,zero_volume,negative_volume,low_greater_than_high,high_less_than_open,high_less_than_close,low_greater_than_open,low_greater_than_close,total_issues
Ticker,,,,,,,,,,,,,
CHD,0,0,0,0,0,1,0,0,0,0,0,0,1
BKR,0,0,0,0,0,1,0,0,0,0,0,0,1
SBAC,0,0,0,0,0,1,0,0,0,0,0,0,1
WTW,0,0,0,0,0,1,0,0,0,0,0,0,1
XEL,0,0,0,0,0,1,0,0,0,0,0,0,1


Companies with at least one zero-volume day: 24


In [8]:
dtype_report = check_yfinance_dtypes(prices)
print(dtype_report)

          expected_dtype    actual_dtype  is_valid
field                                             
Date          datetime64  datetime64[ms]      True
Open             float64         float64      True
High             float64         float64      True
Low              float64         float64      True
Close            float64         float64      True
Volume  int64 or float64  int64, float64      True


The index is strictly sorted chronologically with **zero duplicate dates**. All data types match expectations, and no impossible price values (negative prices or High/Low/Open/Close inversions) exist. 24 companies exhibit isolated zero-volume trading days.

> **Decision**: Do not delete or impute zero-volume rows to preserve panel structure alignment. Trading eligibility and liquidity filters will be enforced during portfolio universe selection.

## 3. Summary Audit Report and Pipeline Decision

### 3.1 Consolidated Data Quality Summary

Consolidate key metrics into a unified summary table.

In [9]:
quality_summary = create_data_quality_summary(prices)
quality_summary

,metric,value
0,Number of assets,499
1,Number of observations,3773
2,Start date,2010-01-04 00:00:00
3,End date,2024-12-30 00:00:00
4,Duplicated dates,False
5,Chronologically sorted index,True
6,Tickers with data for the full period,419
7,Tickers with partial trading periods,80


### 3.2 Data Cleaning Verdict

> **Decision**: The dataset passes all integrity checks. No rows will be dropped, imputed, or modified. The raw panel is passed directly to the factor engineering stage in its original state.